# Reliable Event Pipeline

**Hiring signal:** Data Engineering

**Dataset:** deterministic committed event fixtures containing duplicates, rejected rows and late-arriving data.

This notebook exposes the actual inputs, pipeline structure, reconciliation evidence and reproducible run path without hiding the engineering work inside notebook-only code.


In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
PROJECT_REL = Path('projects/reliable_event_pipeline')
ROOT = Path.cwd()
if not (ROOT / PROJECT_REL).exists() and (ROOT / 'uni_projects' / PROJECT_REL).exists(): ROOT = ROOT / 'uni_projects'
PROJECT = ROOT / PROJECT_REL
assert PROJECT.exists(), 'Open from the cloned uni_projects repository.'
print('Project:', PROJECT.resolve())


## 1. Inspect the real input fixtures


In [ ]:
for name in ('fixtures/batch_1.csv','fixtures/batch_2.csv'):
    path = PROJECT / name
    df = pd.read_csv(path)
    print(f'\n{name}: {df.shape}')
    display(df)


## 2. Implementation and tests


In [ ]:
for pattern in ('run.py','src/*.py','sql/*.sql','tests/*.py'):
    for path in sorted(PROJECT.glob(pattern)):
        print(f'{path.relative_to(PROJECT)}  ({path.stat().st_size:,} bytes)')


## 3. Verified pipeline evidence


In [ ]:
evidence = json.loads((PROJECT / 'results/verified_run.json').read_text(encoding='utf-8'))
print(json.dumps(evidence, indent=2)[:12000])


## 4. Reproduce


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print('Rebuild: cd projects/reliable_event_pipeline && python run.py')
    print('Tests: python -m pytest -q projects/reliable_event_pipeline/tests')


## 5. Interview discussion

Explain schema contracts, reject/quarantine handling, deduplication, late-arriving data, idempotency, audit metrics and SQL reconciliation. The important signal is that rerunning the same batch does not silently corrupt downstream state.
